# 21 cm – Galaxy Cross-Correlation: Uncertainty Budget
### Implementation of Equations 15–17 from La Plante et al. (2023)

**Paper:** *Prospects for 21 cm–Galaxy Cross-Correlations with HERA and the Roman High-Latitude Survey*  
arXiv: [2205.09770](https://arxiv.org/abs/2205.09770)

---

## Physical Background

During the **Epoch of Reionization (EoR)**, neutral hydrogen (H I) in the intergalactic medium (IGM)
emits/absorbs 21 cm radiation.  On large spatial scales, the 21 cm brightness-temperature field and
the galaxy number-density field are **anti-correlated**: overdense regions host the galaxies that
ionise their surroundings, leaving them 21 cm-dark, while underdense regions remain neutral and
hence 21 cm-bright.

The cross-power spectrum $P_{21\times\mathrm{gal}}(k_\perp, k_\parallel)$ quantifies this
anti-correlation as a function of the Fourier wavenumber components perpendicular ($k_\perp$) and
parallel ($k_\parallel$) to the line of sight.  Measuring it is one of the key science goals of
HERA + Roman Space Telescope.

### Notation

| Symbol | Meaning |
|--------|---------|
| $k_\perp$, $k_\parallel$ | Fourier wavenumbers perpendicular / parallel to LOS (h/Mpc) |
| $k = \sqrt{k_\perp^2+k_\parallel^2}$ | Total wavenumber |
| $P_{21}$ | 21 cm auto-power spectrum  $[\mathrm{mK}^2\,(h^{-1}\mathrm{Mpc})^3]$ |
| $P_{\mathrm{gal}}$ | Galaxy auto-power spectrum $[(h^{-1}\mathrm{Mpc})^3]$ |
| $P_{21\times\mathrm{gal}}$ | 21 cm × galaxy cross-power spectrum |
| $P_{21}^{\mathrm{noise}}$ | 21 cm thermal-noise power spectrum (Eq. 11 of paper) |
| $P_{\mathrm{gal}}^{\mathrm{noise}}$ | Galaxy shot-noise power spectrum |
| $T_0(z)$ | Brightness-temperature scaling factor (Eq. 6) |
| $\sigma^2_{21,\mathrm{gal}}$ | Variance of cross-spectrum estimator (Eq. 15) |
| $\sigma^2_{21}$ | Variance of 21 cm auto-spectrum estimator (Eq. 16) |
| $\sigma^2_{\mathrm{gal}}$ | Variance of galaxy auto-spectrum estimator (Eq. 17) |

---


## 1  Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── plotting defaults ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 12,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.35,
})


## 2  Brightness-temperature scaling factor $T_0(z)$ — Equation (6)

$$
T_0(z) = 26\,\frac{T_S - T_\gamma}{T_S}
          \left(\frac{\Omega_b h^2}{0.022}\right)
          \left(\frac{0.143}{\Omega_m h^2}\,\frac{1+z}{10}\right)^{1/2}
          \quad [\text{mK}]
$$

$T_0(z)$ converts dimensionless 21 cm fluctuations from simulations into milli-Kelvin brightness
temperatures.  In the **saturated spin-temperature limit** ($T_S \gg T_\gamma$), the
prefactor $(T_S - T_\gamma)/T_S \rightarrow 1$, which is the standard assumption made
throughout the paper (and below).

**Cosmological parameters** follow Planck 2018 (consistent with the paper).


In [ ]:
# ── Planck 2018 cosmological parameters ──────────────────────────────────────
# (Planck Collaboration 2020, Table 2: TT,TE,EE+lowE+lensing)
Omega_b_h2 = 0.02237   # physical baryon density Omega_b * h^2
Omega_m_h2 = 0.1430    # physical matter density Omega_m * h^2
h_cosmo    = 0.6736    # dimensionless Hubble constant (H0 = 100 h km/s/Mpc)


def T0(z, Omega_b_h2=Omega_b_h2, Omega_m_h2=Omega_m_h2, high_Ts=True):
    '''
    Brightness-temperature scaling factor T0(z) in milli-Kelvin.

    La Plante et al. (2023), Equation (6); originally from Madau et al. (1997).

    In the saturated-spin-temperature limit (T_S >> T_gamma, high_Ts=True)
    the (T_S - T_gamma)/T_S prefactor is set to 1.

    Parameters
    ----------
    z          : float or ndarray  -- cosmological redshift
    Omega_b_h2 : float  -- physical baryon density (Omega_b * h^2)
    Omega_m_h2 : float  -- physical matter density (Omega_m * h^2)
    high_Ts    : bool   -- if True use the saturated spin-temp. approximation

    Returns
    -------
    T0 : float or ndarray  [mK]
    '''
    spin_factor = 1.0 if high_Ts else NotImplemented  # extend for cold reionization
    return (26.0
            * spin_factor
            * (Omega_b_h2 / 0.022)
            * np.sqrt((0.143 / Omega_m_h2) * (1.0 + z) / 10.0))


# ── Evaluate T0 across EoR redshifts ─────────────────────────────────────────
z_arr = np.linspace(6, 12, 200)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z_arr, T0(z_arr), lw=2, color="steelblue")
ax.set_xlabel("Redshift  $z$")
ax.set_ylabel("$T_0(z)$  [mK]")
ax.set_title("Brightness-temperature scaling factor (Eq. 6)")
plt.tight_layout()
plt.show()

print(f"T0 at z = 8  : {T0(8):.2f} mK")
print(f"T0 at z = 10 : {T0(10):.2f} mK")


## 3  Proxy power spectra

Because we do not have simulation output here, we replace the true power spectra with
**arbitrary mathematical models** that reproduce their qualitative features:

* **Power-law shape** — cosmological power spectra are approximately power laws in $k$ on
  the scales of interest.
* **Anti-correlation** — $P_{21\times\mathrm{gal}}$ is *negative* on large scales (the
  21 cm and galaxy fields are anti-correlated during reionization).
* **Noise rising at high $k$** — 21 cm thermal noise grows towards smaller scales.

### Signal power spectra (arbitrary power laws)

$$
P_{21}(k_\perp, k_\parallel)      = A_{21}\, k^{-\alpha_{21}}
\qquad
P_{\mathrm{gal}}(k_\perp, k_\parallel) = A_{\mathrm{gal}}\, k^{-\alpha_{\mathrm{gal}}}
\qquad
P_{21\times\mathrm{gal}} = -A_{\times}\, k^{-\alpha_{\times}}
$$

where $k = \sqrt{k_\perp^2 + k_\parallel^2}$.

### Noise power spectra

**21 cm thermal noise** (proxy for Eq. 11 of the paper, which gives the full interferometer
noise model):

$$P_{21}^{\mathrm{noise}}(k_\perp, k_\parallel) = N_{21}\, k^{+\beta_{21}}$$

The positive slope ($\beta_{21} > 0$) captures the physical expectation that thermal noise
rises at smaller scales (higher $k$).

**Galaxy shot noise** — Poisson noise from the discrete sampling of the galaxy field.
In the limit of a large survey volume this is white noise with amplitude $1/\bar{n}$,
where $\bar{n}$ is the mean comoving galaxy number density:

$$P_{\mathrm{gal}}^{\mathrm{noise}} = \frac{1}{\bar{n}}$$


In [ ]:
# ── Proxy model parameters ────────────────────────────────────────────────────
# These are chosen to give realistic-looking spectra.
# Replace with simulation outputs (e.g. 21cmFAST) for a physical result.

# Signal amplitudes and spectral slopes
A_21       = 1.5e3   # 21 cm auto  [mK^2 (h^-1 Mpc)^3]
alpha_21   = 2.0     # slope; CDM matter P(k) ~ k^-3 => brightness T P ~ k^-2

A_gal      = 5.0e2   # galaxy auto  [(h^-1 Mpc)^3]
alpha_gal  = 2.2     # slightly steeper (galaxies are more biased tracers)

A_cross    = 8.0e2   # 21cm x galaxy  [mK (h^-1 Mpc)^3]  (enters with minus sign)
alpha_cross = 2.1

# Noise amplitudes
N_21       = 2.0e4   # 21 cm thermal noise amplitude  [mK^2 (h^-1 Mpc)^3]
beta_21    = 0.5     # positive slope: noise rises at high k

n_bar_gal  = 1.0e-3  # mean galaxy number density  [h^3 Mpc^-3]


# ── 2-D grid in (k_perp, k_par) ──────────────────────────────────────────────
N = 200
k_perp_1d = np.logspace(-2, 0, N)   # h/Mpc, perpendicular to line of sight
k_par_1d  = np.logspace(-2, 0, N)   # h/Mpc, parallel to line of sight

# Broadcast to shape (N, N) using 'ij' indexing
k_perp, k_par = np.meshgrid(k_perp_1d, k_par_1d, indexing="ij")
k_tot = np.sqrt(k_perp**2 + k_par**2)   # |k| at every grid point


# ── Helper: isotropic power law ───────────────────────────────────────────────
def power_law(k, amplitude, slope):
    '''Isotropic power-law spectrum: P(k) = amplitude * k^(-slope).'''
    return amplitude * k**(-slope)


# ── Signal spectra ────────────────────────────────────────────────────────────
P21     = power_law(k_tot, A_21,    alpha_21)          # 21 cm auto
P_gal   = power_law(k_tot, A_gal,   alpha_gal)         # galaxy auto
P21_gal = -power_law(k_tot, A_cross, alpha_cross)       # cross  (negative!)

# ── Noise spectra ─────────────────────────────────────────────────────────────
# 21 cm thermal noise: positive slope => noise grows at small scales
P21_noise   = power_law(k_tot, N_21, -beta_21)          # note: slope flipped
# Galaxy shot noise: white noise (k-independent), amplitude = 1/n_bar
P_gal_noise = np.full_like(k_tot, 1.0 / n_bar_gal)

# ── Quick sanity check ────────────────────────────────────────────────────────
idx = (N // 4, N // 4)
print("Proxy power-spectrum values at k_perp = k_par = 0.1 h/Mpc:")
print(f"  P21           = {P21[idx]:.2e}  mK^2 (h^-1 Mpc)^3")
print(f"  P_gal         = {P_gal[idx]:.2e}  (h^-1 Mpc)^3")
print(f"  P21_gal       = {P21_gal[idx]:.2e}  mK (h^-1 Mpc)^3  [negative = anti-corr.]")
print(f"  P21_noise     = {P21_noise[idx]:.2e}  mK^2 (h^-1 Mpc)^3")
print(f"  P_gal_noise   = {P_gal_noise[idx]:.2e}  (h^-1 Mpc)^3  [shot noise = 1/n_bar]")


### 3.1  Visualise the proxy power spectra

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    (P21,             r"$P_{21}(k_\perp, k_\parallel)$",              "viridis"),
    (P_gal,           r"$P_{\rm gal}(k_\perp, k_\parallel)$",        "plasma"),
    (np.abs(P21_gal), r"$|P_{21\times{\rm gal}}(k_\perp, k_\parallel)|$","inferno"),
]

for ax, (data, title, cmap) in zip(axes, panels):
    im = ax.pcolormesh(k_perp_1d, k_par_1d, np.log10(data.T),
                       cmap=cmap, shading="auto")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(r"$\log_{10}$[amplitude]")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k_\perp$  [h Mpc$^{-1}$]")
    ax.set_ylabel(r"$k_\parallel$  [h Mpc$^{-1}$]")
    ax.set_title(title)

plt.suptitle("Proxy signal power spectra  (log10 colour scale)", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Fix the observing redshift to z = 8 (mid-reionization in the paper's fiducial model)
z_obs  = 8.0
T0_val = T0(z_obs)
print(f"Evaluating uncertainties at z = {z_obs}")
print(f"T0(z={z_obs}) = {T0_val:.3f} mK")


## 4  Variance of the galaxy power spectrum — Equation (17)

$$
\sigma^2_{\mathrm{gal}}
= \mathrm{var}\!\left[P_{\mathrm{gal}}(k_\perp, k_\parallel)\right]
= \Bigl[P_{\mathrm{gal}}(k_\perp, k_\parallel)
         + P^{\mathrm{noise}}_{\mathrm{gal}}(k_\parallel)\Bigr]^2
$$

This is the **cosmic-variance + shot-noise** formula for a Gaussian random field.
In the signal-dominated limit ($P_{\mathrm{gal}} \gg P^{\mathrm{noise}}_{\mathrm{gal}}$),
$\sigma_{\mathrm{gal}} \approx P_{\mathrm{gal}}$ (cosmic variance).  In the shot-noise
limit ($P^{\mathrm{noise}}_{\mathrm{gal}} \gg P_{\mathrm{gal}}$),
$\sigma_{\mathrm{gal}} \approx 1/\bar{n}$.

> **Note on mode counting:** Equations (15–17) are written for a *single* $(k_\perp, k_\parallel)$
> mode.  After averaging over $N_k$ independent modes in a bin the variance divides by $N_k$.
> We focus on the per-mode form here.


In [ ]:
def sigma2_gal(P_gal, P_gal_noise):
    '''
    Per-mode variance of the galaxy auto-power-spectrum estimator.

    La Plante et al. (2023), Equation (17):

        sigma2_gal = [P_gal(k_perp, k_par) + P_gal_noise(k_par)]^2

    Parameters
    ----------
    P_gal       : array  -- galaxy signal power spectrum
    P_gal_noise : array  -- galaxy shot-noise power spectrum (= 1/n_bar)

    Returns
    -------
    sigma2 : array, same shape as inputs
    '''
    return (P_gal + P_gal_noise)**2


sig2_gal_arr = sigma2_gal(P_gal, P_gal_noise)
sig_gal_arr  = np.sqrt(sig2_gal_arr)   # standard deviation

print(f"sigma_gal at (k_perp, k_par) = (0.1, 0.1) h/Mpc : {sig_gal_arr[N//4, N//4]:.2e}")


## 5  Variance of the 21 cm power spectrum — Equation (16)

$$
\sigma^2_{21}
= \mathrm{var}\!\left[P_{21}(k_\perp, k_\parallel)\right]
= \left[
      \frac{P_{21}(k_\perp, k_\parallel)}{T_0(z)^2}
    + \frac{P^{\mathrm{noise}}_{21}(k_\perp, k_\parallel)}{T_0(z)^2}
  \right]^2
$$

The $1/T_0^2$ factors convert from the simulation's brightness-temperature units to the
dimensionless fluctuations.  At high redshift, $T_0(z)$ is larger (see §2), which *suppresses*
the noise contribution relative to the signal — a favourable scaling for high-$z$ 21 cm
experiments.

$P^{\mathrm{noise}}_{21}$ (Eq. 11 of the paper) encodes the **thermal noise** of the HERA
interferometer; it grows steeply towards small scales (high $k$), eventually dominating the
error budget.


In [ ]:
def sigma2_21(P21, P21_noise, T0_val):
    '''
    Per-mode variance of the 21 cm auto-power-spectrum estimator.

    La Plante et al. (2023), Equation (16):

        sigma2_21 = [(P21 + P21_noise) / T0^2]^2

    Parameters
    ----------
    P21       : array  -- 21 cm signal power spectrum  [mK^2 (h^-1 Mpc)^3]
    P21_noise : array  -- 21 cm thermal-noise spectrum [mK^2 (h^-1 Mpc)^3]
    T0_val    : float  -- T0(z)  [mK]

    Returns
    -------
    sigma2 : array, same shape as inputs
    '''
    return ((P21 + P21_noise) / T0_val**2)**2


sig2_21_arr = sigma2_21(P21, P21_noise, T0_val)
sig_21_arr  = np.sqrt(sig2_21_arr)   # standard deviation

print(f"T0(z={z_obs})^2 = {T0_val**2:.2f}  mK^2")
print(f"sigma_21 at (k_perp, k_par) = (0.1, 0.1) h/Mpc : {sig_21_arr[N//4, N//4]:.2e}")


## 6  Variance of the 21 cm × galaxy cross-power spectrum — Equation (15)

$$
\sigma^2_{21,\mathrm{gal}}
= \mathrm{var}\!\left[\frac{P_{21\times\mathrm{gal}}(k_\perp, k_\parallel)}{T_0(z)}\right]
= \frac{1}{2}\!\left[
      P^2_{21\times\mathrm{gal}}(k_\perp, k_\parallel)
    + \sigma_{21}(k_\perp, k_\parallel)\,\sigma_{\mathrm{gal}}(k_\perp, k_\parallel)
  \right]
$$

This follows from the **Isserlis–Wick theorem** applied to the four-point function of a
joint Gaussian field.  The two terms have distinct physical origins:

1. **Cross-spectrum squared** — cosmic variance: even in the absence of noise, the estimator
   scatters because we observe only a finite number of independent modes in the universe.
2. **Product of auto-spectrum standard deviations** — noise cross-contamination: thermal and
   shot noise from each individual field leak into the cross measurement.

At low $k$ (large scales), term 1 typically dominates.  At high $k$, term 2 dominates because
$\sigma_{21}$ (thermal noise) and/or $\sigma_{\mathrm{gal}}$ (shot noise) become large.


In [ ]:
def sigma2_21_gal(P21_gal, sig_21, sig_gal, T0_val):
    '''
    Per-mode variance of the 21 cm x galaxy cross-power-spectrum estimator.

    La Plante et al. (2023), Equation (15).
    Derived originally in Lidz et al. (2009).

    The cross-spectrum P21_gal is defined in simulation units [mK (h^-1 Mpc)^3];
    the function divides by T0 internally to match the normalisation in Eq. 15.

        sigma2_cross = 0.5 * [(P21_gal/T0)^2 + sigma_21 * sigma_gal]

    Parameters
    ----------
    P21_gal : array  -- 21 cm x galaxy cross-spectrum  [mK (h^-1 Mpc)^3]
    sig_21  : array  -- std dev of 21 cm auto-spectrum  (sqrt of sigma2_21)
    sig_gal : array  -- std dev of galaxy auto-spectrum (sqrt of sigma2_gal)
    T0_val  : float  -- T0(z)  [mK]

    Returns
    -------
    sigma2 : array, same shape as inputs
    '''
    # Normalise cross-spectrum by T0 to go to dimensionless fluctuation units
    P21_gal_norm = P21_gal / T0_val
    return 0.5 * (P21_gal_norm**2 + sig_21 * sig_gal)


sig2_cross_arr = sigma2_21_gal(P21_gal, sig_21_arr, sig_gal_arr, T0_val)
sig_cross_arr  = np.sqrt(sig2_cross_arr)

print(f"sigma_cross at (k_perp, k_par) = (0.1, 0.1) h/Mpc : {sig_cross_arr[N//4, N//4]:.2e}")

# ── Break down the two terms of Eq. 15 ───────────────────────────────────────
term1 = 0.5 * (P21_gal / T0_val)**2        # cosmic variance of cross-spectrum
term2 = 0.5 * sig_21_arr * sig_gal_arr     # noise cross-contamination

ratio = term1 / (term1 + term2)
print(f"Fraction of sigma^2 from term1 (cosmic var.) at k=(0.1,0.1): {ratio[N//4, N//4]:.2%}")


## 7  Signal-to-noise ratio

The per-mode SNR of the cross-spectrum detection is

$$
\mathrm{SNR}(k_\perp, k_\parallel)
= \frac{\left|P_{21\times\mathrm{gal}}(k_\perp, k_\parallel)\,/\,T_0(z)\right|}
         {\sigma_{21,\mathrm{gal}}(k_\perp, k_\parallel)}
$$

Summing in quadrature over all independent modes gives the total detection significance:

$$
\mathrm{SNR}_{\mathrm{total}} = \sqrt{\sum_{k} \mathrm{SNR}^2(k_\perp, k_\parallel)}
$$


In [ ]:
# Per-mode signal-to-noise ratio
SNR = np.abs(P21_gal / T0_val) / sig_cross_arr

# Total significance (sum in quadrature)
total_SNR = np.sqrt(np.sum(SNR**2))

print(f"Peak per-mode SNR          : {SNR.max():.2f}")
print(f"SNR at k=(0.1,0.1) h/Mpc  : {SNR[N//4, N//4]:.2f}")
print(f"Total significance (all modes, proxy grid) : {total_SNR:.1f} sigma")


## 8  Diagnostic 2-D maps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

panels = [
    (np.abs(P21_gal / T0_val),
     r"$|P_{21\times{\rm gal}}|/T_0$  (signal, log10)", "RdBu_r"),
    (sig_21_arr,
     r"$\sigma_{21}$  — 21 cm auto std dev (log10)", "hot"),
    (sig_gal_arr,
     r"$\sigma_{\rm gal}$  — galaxy auto std dev (log10)", "cool"),
    (sig_cross_arr,
     r"$\sigma_{21,{\rm gal}}$  — cross std dev (log10)", "viridis"),
    (SNR,
     r"SNR$(k_\perp, k_\parallel)$  (log10)", "plasma"),
    (np.clip(SNR, 0, 5),
     r"SNR  (linear, clipped at 5)", "seismic"),
]

for ax, (data, title, cmap) in zip(axes.flat, panels):
    if "linear" in title:
        im = ax.pcolormesh(k_perp_1d, k_par_1d, data.T,
                           cmap=cmap, shading="auto", vmin=0, vmax=5)
        cb = fig.colorbar(im, ax=ax)
        cb.set_label("SNR")
    else:
        im = ax.pcolormesh(k_perp_1d, k_par_1d, np.log10(data.T),
                           cmap=cmap, shading="auto")
        cb = fig.colorbar(im, ax=ax)
        cb.set_label("log10[value]")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$k_\perp$  [h Mpc$^{-1}$]")
    ax.set_ylabel(r"$k_\parallel$  [h Mpc$^{-1}$]")
    ax.set_title(title, fontsize=10)

plt.suptitle(
    f"Uncertainty budget at z = {z_obs},  T0 = {T0_val:.1f} mK",
    y=1.01, fontsize=14
)
plt.tight_layout()
plt.show()


## 9  1-D slices along the diagonal $k_\perp = k_\parallel$

Cutting along the diagonal $k_\perp = k_\parallel = k/\sqrt{2}$ gives a clean 1-D view of
how each uncertainty term scales with total wavenumber $k$.  At large scales (small $k$) the
cross-signal dominates; at small scales, thermal and shot noise take over.


In [ ]:
# The grid is square and log-spaced, so diagonal indices simply match
diag_idx = np.arange(N)
k_diag   = k_tot[diag_idx, diag_idx]

signal_diag    = np.abs(P21_gal[diag_idx, diag_idx] / T0_val)
sig_21_diag    = sig_21_arr[diag_idx, diag_idx]
sig_gal_diag   = sig_gal_arr[diag_idx, diag_idx]
sig_cross_diag = sig_cross_arr[diag_idx, diag_idx]
SNR_diag       = SNR[diag_idx, diag_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: amplitudes ────────────────────────────────────────────────────
ax1.loglog(k_diag, signal_diag,    lw=2.5, label=r"$|P_{21\times{\rm gal}}|/T_0$  (signal)")
ax1.loglog(k_diag, sig_21_diag,    lw=2, ls="--",
           label=r"$\sigma_{21}$  (21 cm noise)")
ax1.loglog(k_diag, sig_gal_diag,   lw=2, ls="-.",
           label=r"$\sigma_{\rm gal}$  (galaxy noise)")
ax1.loglog(k_diag, sig_cross_diag, lw=2, ls=":",
           label=r"$\sigma_{21,{\rm gal}}$  (total cross uncertainty)")
ax1.set_xlabel(r"$k$  [h Mpc$^{-1}$]  ($k_\perp = k_\parallel$ diagonal)")
ax1.set_ylabel("Amplitude  (arbitrary units)")
ax1.set_title("Signal and uncertainty amplitudes")
ax1.legend(loc="best")

# ── Right panel: SNR ──────────────────────────────────────────────────────────
ax2.semilogx(k_diag, SNR_diag, lw=2.5, color="darkorange")
ax2.axhline(1, color="grey", ls="--", lw=1.2, label="SNR = 1  (detection threshold)")
ax2.fill_between(k_diag, SNR_diag, 1,
                 where=(SNR_diag >= 1),
                 alpha=0.25, color="green", label="Detected modes (SNR > 1)")
ax2.set_xlabel(r"$k$  [h Mpc$^{-1}$]")
ax2.set_ylabel("Per-mode SNR")
ax2.set_title("Signal-to-noise ratio along diagonal")
ax2.legend()

plt.tight_layout()
plt.show()


## 10  Redshift evolution of the uncertainty at a fixed mode

$T_0(z)$ grows with redshift (see §2), so the 21 cm thermal-noise contribution
($\propto 1/T_0^2$) is *suppressed* at higher $z$.  Here we track all three
uncertainties as a function of redshift at a single representative mode.


In [ ]:
# Representative mode indices
i_rep, j_rep = N // 3, N // 3
k_rep_perp   = k_perp_1d[i_rep]
k_rep_par    = k_par_1d[j_rep]
print(f"Representative mode: k_perp = {k_rep_perp:.3f},  k_par = {k_rep_par:.3f}  h/Mpc")

z_range = np.linspace(6, 12, 100)

sig_cross_z = []
sig_21_z    = []
sig_gal_z   = []

for z_val in z_range:
    T0_z = T0(z_val)

    # 21 cm uncertainty (depends on z through T0)
    s21  = np.sqrt(sigma2_21(P21[i_rep, j_rep], P21_noise[i_rep, j_rep], T0_z))
    # Galaxy uncertainty (z-independent in this proxy model)
    sgal = np.sqrt(sigma2_gal(P_gal[i_rep, j_rep], P_gal_noise[i_rep, j_rep]))
    # Cross uncertainty
    scr  = np.sqrt(sigma2_21_gal(P21_gal[i_rep, j_rep], s21, sgal, T0_z))

    sig_21_z.append(s21)
    sig_gal_z.append(sgal)
    sig_cross_z.append(scr)

sig_21_z    = np.array(sig_21_z)
sig_gal_z   = np.array(sig_gal_z)
sig_cross_z = np.array(sig_cross_z)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(z_range, sig_cross_z, lw=2.5,
            label=r"$\sigma_{21,{\rm gal}}$  (cross)")
ax.semilogy(z_range, sig_21_z,    lw=2, ls="--",
            label=r"$\sigma_{21}$  (21 cm auto)")
ax.semilogy(z_range, sig_gal_z,   lw=2, ls="-.",
            label=r"$\sigma_{\rm gal}$  (galaxy auto, z-independent in proxy)")
ax.set_xlabel("Redshift  $z$")
ax.set_ylabel("Uncertainty amplitude")
ax.set_title(
    rf"Redshift evolution at $k_\perp={k_rep_perp:.2f}$,  "
    rf"$k_\parallel={k_rep_par:.2f}$  h/Mpc"
)
ax.legend()
plt.tight_layout()
plt.show()


## 11  Summary and next steps

This notebook implements the **three uncertainty equations** from La Plante et al. (2023),
using arbitrary power-law proxy spectra in place of actual simulations.

| Equation | Quantity | Physics |
|----------|----------|---------|
| (6)  | $T_0(z)$ | Converts simulation units to mK; grows as $(1+z)^{1/2}$ |
| (17) | $\sigma^2_{\rm gal}$ | Shot-noise-dominated at $k \gtrsim 0.3\,h/$Mpc |
| (16) | $\sigma^2_{21}$ | Thermal-noise-dominated at high $k$; improves at high $z$ via $T_0$ |
| (15) | $\sigma^2_{21,{\rm gal}}$ | Combines both: cosmic-variance at low $k$, noise at high $k$ |

### To make this fully physical

1. Replace the `power_law(...)` calls in **Section 3** with outputs from a proper EoR
   simulation, e.g.:
   - [21cmFAST](https://github.com/21cmfast/21cmFAST) for the 21 cm field
   - The paper's own public code:
     [github.com/plaplant/21cm_gal_cross_correlation](https://github.com/plaplant/21cm_gal_cross_correlation)
2. Use the full interferometer-noise model (Eq. 11) for $P_{21}^{\rm noise}$ — it encodes
   HERA's baseline distribution and integration time.
3. Apply the **foreground wedge** mask: modes with
   $k_\parallel < k_\perp \sin\theta_{\rm FoV}$ are typically excised.
4. Sum the per-mode SNR over all *unmasked* modes in the observed $(k_\perp, k_\parallel)$
   space to forecast the total detection significance.

### References

- La Plante et al. (2023), ApJ — [arXiv:2205.09770](https://arxiv.org/abs/2205.09770)
- Lidz et al. (2009) — original derivation of Equations (15–17)
- Madau et al. (1997) — derivation of Equation (6) for $T_0(z)$
